In [1]:
print("Hello World")

Hello World


In [2]:
from dotenv import load_dotenv
load_dotenv()


True

In [3]:
import pandas as pd

# QA
inputs = [
    "For customer-facing applications, which company's models dominate the top rankings?",
    "What percentage of respondents are using RAG in some form?",
    "How often are most respondents updating their models?",
]

outputs = [
    "OpenAI models dominate, with 3 of the top 5 and half of the top 10 most popular models for customer-facing apps.",
    "70% of respondents are using RAG in some form.",
    "More than 50% update their models at least monthly, with 17% doing so weekly.",
]

# Dataset
qa_pairs = [{"question": q, "answer": a} for q, a in zip(inputs, outputs)]
df = pd.DataFrame(qa_pairs)

# Write to csv
csv_path = "/Users/palla/Desktop/LLMOPS/data/goldens.csv"
df.to_csv(csv_path, index=False)

In [4]:
from langsmith import Client

client = Client()
dataset_name = "AgenticAIReportGoldens"

# Store
dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Input and expected output pairs for AgenticAIReport",
)
client.create_examples(
    inputs=[{"question": q} for q in inputs],
    outputs=[{"answer": a} for a in outputs],
    dataset_id=dataset.id,
)

{'example_ids': ['2a917879-b7d7-4bc9-ab48-f249c9485b30',
  'c0404d91-87dc-4553-9296-ce882cc3d17b',
  'd468a780-549a-49ac-9d1f-c92badd172c5'],
 'count': 3,
 'as_of': '2026-06-28T19:12:49.247542608Z'}

In [5]:
import sys
sys.path.append("/Users/palla/Desktop/LLMOPS")

from pathlib import Path
from multi_doc_chat.src.document_ingestion.data_ingestion import ChatIngestor
from multi_doc_chat.src.document_chat.retrieval import ConversationalRAG
import os

# Simple file adapter for local file paths
class LocalFileAdapter:
    """Adapter for local file paths to work with ChatIngestor."""
    def __init__(self, file_path: str):
        self.path = Path(file_path)
        self.name = self.path.name
    
    def getbuffer(self) -> bytes:
        return self.path.read_bytes()


def answer_ai_report_question(
    inputs: dict,
    data_path: str = "/Users/palla/Desktop/LLMOPS/data/Synthetic_2025_AI_Engineering_Report.txt",
    chunk_size: int = 1000,
    chunk_overlap: int = 200,
    k: int = 5
) -> dict:
    """
    Answer questions about the AI Engineering Report using RAG.
    
    Args:
        inputs: Dictionary containing the question, e.g., {"question": "What is RAG?"}
        data_path: Path to the AI Engineering Report text file
        chunk_size: Size of text chunks for splitting
        chunk_overlap: Overlap between chunks
        k: Number of documents to retrieve
    
    Returns:
        Dictionary with the answer, e.g., {"answer": "RAG stands for..."}
    """
    try:
        # Extract question from inputs
        question = inputs.get("question", "")
        if not question:
            return {"answer": "No question provided"}
        
        # Check if file exists
        if not Path(data_path).exists():
            return {"answer": f"Data file not found: {data_path}"}
        
        # Create file adapter
        file_adapter = LocalFileAdapter(data_path)
        
        # Build index using ChatIngestor
        ingestor = ChatIngestor(
            temp_base="data",
            faiss_base="faiss_index",
            use_session_dirs=True
        )
        
        # Build retriever
        ingestor.built_retriver(
            uploaded_files=[file_adapter],
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            k=k
        )
        
        # Get session ID and index path
        session_id = ingestor.session_id
        index_path = f"faiss_index/{session_id}"
        
        # Create RAG instance and load retriever
        rag = ConversationalRAG(session_id=session_id)
        rag.load_retriever_from_faiss(
            index_path=index_path,
            k=k,
            index_name=os.getenv("FAISS_INDEX_NAME", "index")
        )
        
        # Get answer
        answer = rag.invoke(question, chat_history=[])
        
        return {"answer": answer}
        
    except Exception as e:
        return {"answer": f"Error: {str(e)}"}


In [ ]:
# Test the function with a sample question
test_input = {"question": "what are the emerging skills an AI engineer should have?"}
result = answer_ai_report_question(test_input)
print("Question:", test_input["question"])
print("\nAnswer:", result["answer"])


In [14]:
from langsmith.evaluation import evaluate, LangChainStringEvaluator


ImportError: cannot import name 'LangChainStringEvaluator' from 'langsmith.evaluation' (c:\Users\palla\Desktop\LLMOPS\.venv\Lib\site-packages\langsmith\evaluation\__init__.py)

In [9]:
# Example: Test with all golden questions
print("Testing all questions from the dataset:\n")
for i, q in enumerate(inputs, 1):
    test_input = {"question": q}
    result = answer_ai_report_question(test_input)
    print(f"Q{i}: {q}")
    print(f"A{i}: {result['answer']}\n")
    print("-" * 80 + "\n")


{"session": "session_20260629_094629_7ae539", "timestamp": "2026-06-29T04:16:29.200418Z", "level": "info", "event": "ChatIngestor ready"}
{"uploaded": "Synthetic_2025_AI_Engineering_Report.txt", "saved_as": "data\\session_20260629_094629_7ae539\\8dbb19ce.txt", "timestamp": "2026-06-29T04:16:29.205443Z", "level": "info", "event": "File saved for ingestion"}
{"count": 1, "timestamp": "2026-06-29T04:16:29.239054Z", "level": "info", "event": "Documents loaded"}


Testing all questions from the dataset:



{"added": 2, "timestamp": "2026-06-29T04:16:31.981012Z", "level": "info", "event": "Ingestion complete"}
{"session_id": "session_20260629_094629_7ae539", "timestamp": "2026-06-29T04:16:32.010240Z", "level": "info", "event": "LLM loaded successfully"}
{"session_id": "session_20260629_094629_7ae539", "timestamp": "2026-06-29T04:16:32.014889Z", "level": "info", "event": "ConversationalRAG initialized"}
{"session_id": "session_20260629_094629_7ae539", "timestamp": "2026-06-29T04:16:32.112769Z", "level": "info", "event": "LCEL graph built successfully"}
{"index_path": "faiss_index/session_20260629_094629_7ae539", "index_name": "index", "search_type": "mmr", "k": 5, "fetch_k": 20, "lambda_mult": 0.5, "session_id": "session_20260629_094629_7ae539", "timestamp": "2026-06-29T04:16:32.113761Z", "level": "info", "event": "FAISS retriever loaded successfully"}
{"session_id": "session_20260629_094629_7ae539", "user_input": "For customer-facing applications, which company's models dominate the top r

Q1: For customer-facing applications, which company's models dominate the top rankings?
A1: I don't know.

--------------------------------------------------------------------------------



{"added": 2, "timestamp": "2026-06-29T04:16:39.549096Z", "level": "info", "event": "Ingestion complete"}
{"session_id": "session_20260629_094637_3cb4e7", "timestamp": "2026-06-29T04:16:39.563108Z", "level": "info", "event": "LLM loaded successfully"}
{"session_id": "session_20260629_094637_3cb4e7", "timestamp": "2026-06-29T04:16:39.564501Z", "level": "info", "event": "ConversationalRAG initialized"}
{"session_id": "session_20260629_094637_3cb4e7", "timestamp": "2026-06-29T04:16:39.651779Z", "level": "info", "event": "LCEL graph built successfully"}
{"index_path": "faiss_index/session_20260629_094637_3cb4e7", "index_name": "index", "search_type": "mmr", "k": 5, "fetch_k": 20, "lambda_mult": 0.5, "session_id": "session_20260629_094637_3cb4e7", "timestamp": "2026-06-29T04:16:39.652759Z", "level": "info", "event": "FAISS retriever loaded successfully"}
{"session_id": "session_20260629_094637_3cb4e7", "user_input": "What percentage of respondents are using RAG in some form?", "answer_previe

Q2: What percentage of respondents are using RAG in some form?
A2: I don't know.

--------------------------------------------------------------------------------



{"added": 2, "timestamp": "2026-06-29T04:16:46.196734Z", "level": "info", "event": "Ingestion complete"}
{"session_id": "session_20260629_094643_818378", "timestamp": "2026-06-29T04:16:46.206186Z", "level": "info", "event": "LLM loaded successfully"}
{"session_id": "session_20260629_094643_818378", "timestamp": "2026-06-29T04:16:46.207183Z", "level": "info", "event": "ConversationalRAG initialized"}
{"session_id": "session_20260629_094643_818378", "timestamp": "2026-06-29T04:16:46.301262Z", "level": "info", "event": "LCEL graph built successfully"}
{"index_path": "faiss_index/session_20260629_094643_818378", "index_name": "index", "search_type": "mmr", "k": 5, "fetch_k": 20, "lambda_mult": 0.5, "session_id": "session_20260629_094643_818378", "timestamp": "2026-06-29T04:16:46.302261Z", "level": "info", "event": "FAISS retriever loaded successfully"}
{"session_id": "session_20260629_094643_818378", "user_input": "How often are most respondents updating their models?", "answer_preview": "

Q3: How often are most respondents updating their models?
A3: The report does not specify how often most respondents update their models. However, it emphasizes "Monitoring and continuous improvement" within the AI Engineering Stack and mentions "CI/CD pipelines" for production deployments, suggesting an ongoing process. For Retrieval-Augmented Generation (RAG) systems, "Continuous index updates" are also highlighted.

--------------------------------------------------------------------------------



In [18]:
from langsmith.evaluation import evaluate

# Evaluators
qa_evaluator = [correctness_evaluator]
dataset_name = "AgenticAIReportGoldens"

# Run evaluation using our RAG function
experiment_results = evaluate(
    answer_ai_report_question,
    data=dataset_name,
    evaluators=qa_evaluator,
    experiment_prefix="test-agenticAIReport-qa-rag",
    # Experiment metadata
    metadata={
        "variant": "RAG with FAISS and AI Engineering Report",
        "chunk_size": 1000,
        "chunk_overlap": 200,
        "k": 5,
    },
)

View the evaluation results for experiment: 'test-agenticAIReport-qa-rag-25096d66' at:
https://smith.langchain.com/o/2a6ca14a-5688-47f5-9eb8-6bdacf8a6c35/datasets/c307a947-c500-4f91-9add-4326329b48bc/compare?selectedSessions=945dda33-bd78-4e9b-a5da-647609c1b92c




0it [00:00, ?it/s]{"session": "session_20260629_120158_786e4f", "timestamp": "2026-06-29T06:31:58.445352Z", "level": "info", "event": "ChatIngestor ready"}
{"uploaded": "Synthetic_2025_AI_Engineering_Report.txt", "saved_as": "data\\session_20260629_120158_786e4f\\6b5e9eb7.txt", "timestamp": "2026-06-29T06:31:58.448399Z", "level": "info", "event": "File saved for ingestion"}
{"count": 1, "timestamp": "2026-06-29T06:31:58.450345Z", "level": "info", "event": "Documents loaded"}
{"added": 2, "timestamp": "2026-06-29T06:32:01.142107Z", "level": "info", "event": "Ingestion complete"}
{"session_id": "session_20260629_120158_786e4f", "timestamp": "2026-06-29T06:32:01.151632Z", "level": "info", "event": "LLM loaded successfully"}
{"session_id": "session_20260629_120158_786e4f", "timestamp": "2026-06-29T06:32:01.151632Z", "level": "info", "event": "ConversationalRAG initialized"}
{"session_id": "session_20260629_120158_786e4f", "timestamp": "2026-06-29T06:32:01.205795Z", "level": "info", "event"

## Custom Correctness Evaluator

Creating an LLM-as-a-Judge evaluator to assess semantic and factual alignment


In [17]:
from langsmith.schemas import Run, Example
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

def correctness_evaluator(run: Run, example: Example) -> dict:
    """
    Custom LLM-as-a-Judge evaluator for correctness.
    
    Correctness means how well the actual model output matches the reference output 
    in terms of factual accuracy, coverage, and meaning.
    
    Args:
        run: The Run object containing the actual outputs
        example: The Example object containing the expected outputs
    
    Returns:
        dict with 'score' (1 for correct, 0 for incorrect) and 'reasoning'
    """
    # Extract actual and expected outputs
    actual_output = run.outputs.get("answer", "")
    expected_output = example.outputs.get("answer", "")
    input_question = example.inputs.get("question", "")
    
    # Define the evaluation prompt
    eval_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are an evaluator whose job is to judge correctness.

Correctness means how well the actual model output matches the reference output in terms of factual accuracy, coverage, and meaning.

- If the actual output matches the reference output semantically (even if wording differs), it should be marked correct.
- If the output misses key facts, introduces contradictions, or is factually incorrect, it should be marked incorrect.

Do not penalize for stylistic or formatting differences unless they change meaning."""),
        ("human", """<example>
<input>
{input}
</input>

<output>
Expected Output: {expected_output}

Actual Output: {actual_output}
</output>
</example>

Please grade the following agent run given the input, expected output, and actual output.
Focus only on correctness (semantic and factual alignment).

Respond with:
1. A brief reasoning (1-2 sentences)
2. A final verdict: either "CORRECT" or "INCORRECT"

Format your response as:
Reasoning: [your reasoning]
Verdict: [CORRECT or INCORRECT]""")
    ])
    
    # Initialize LLM (using Gemini as shown in your config)
    llm = ChatGoogleGenerativeAI(
        model="gemini-3.5-flash",
        temperature=0
    )
    
    # Create chain and invoke
    chain = eval_prompt | llm
    
    try:
        response = chain.invoke({
            "input": input_question,
            "expected_output": expected_output,
            "actual_output": actual_output
        })
        
        response_text = response.content
        
        # Parse the response
        reasoning = ""
        verdict = ""
        
        for line in response_text.split('\n'):
            if line.startswith("Reasoning:"):
                reasoning = line.replace("Reasoning:", "").strip()
            elif line.startswith("Verdict:"):
                verdict = line.replace("Verdict:", "").strip()
        
        # Convert verdict to score (1 for correct, 0 for incorrect)
        score = 1 if "CORRECT" in verdict.upper() else 0
        
        return {
            "key": "correctness",
            "score": score,
            "reasoning": reasoning,
            "comment": f"Verdict: {verdict}"
        }
        
    except Exception as e:
        return {
            "key": "correctness",
            "score": 0,
            "reasoning": f"Error during evaluation: {str(e)}"
        }


### Run Evaluation with Custom Correctness Evaluator


In [ ]:
# Run evaluation with the custom correctness evaluator
from langsmith.evaluation import evaluate

# Define evaluators - using custom correctness evaluator
evaluators = [correctness_evaluator]

dataset_name = "AgenticAIReportGoldens"

# Run evaluation
experiment_results = evaluate(
    answer_ai_report_question,
    data=dataset_name,
    evaluators=evaluators,
    experiment_prefix="agenticAIReport-correctness-eval",
    description="Evaluating RAG system with custom correctness evaluator (LLM-as-a-Judge)",
    metadata={
        "variant": "RAG with FAISS and AI Engineering Report",
        "evaluator": "custom_correctness_llm_judge",
        "model": "gemini-2.5-pro",
        "chunk_size": 1000,
        "chunk_overlap": 200,
        "k": 5,
    },
)

print("\nEvaluation completed! Check the LangSmith UI for detailed results.")


### Optional: Combine Multiple Evaluators

You can use multiple evaluators together to get different perspectives on your RAG system's performance.


In [ ]:
# Example: Combine custom correctness evaluator with LangChain's built-in evaluators
from langsmith.evaluation import evaluate, LangChainStringEvaluator

# Combine custom and built-in evaluators
combined_evaluators = [
    correctness_evaluator,  # Custom LLM-as-a-Judge
    LangChainStringEvaluator("cot_qa"),  # Chain-of-thought QA evaluator
]

# Run evaluation with multiple evaluators
# Uncomment to run:
# experiment_results_combined = evaluate(
#     answer_ai_report_question,
#     data=dataset_name,
#     evaluators=combined_evaluators,
#     experiment_prefix="agenticAIReport-multi-eval",
#     description="Evaluating RAG system with multiple evaluators",
#     metadata={
#         "variant": "RAG with FAISS",
#         "evaluators": "correctness + cot_qa",
#         "chunk_size": 1000,
#         "chunk_overlap": 200,
#         "k": 5,
#     },
# )
